# CIFAR-100 transform comparison

This notebook selects one random CIFAR-100 image and compares the two transform pipelines:

1. Native CIFAR resolution: padding and random crop at `32x32`.
2. ImageNet-style resolution: bicubic resize and random crop at `224x224`.

The dataset is loaded with `download=False` and the images are unnormalized before plotting.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision import datasets, transforms

DATASET_PATH = Path(r"F:/Dataset/CIFAR-100/")
SEED = 2023
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dataset path: {DATASET_PATH}")
print(f"Device: {device}")

In [ ]:
def create_transforms_32():
    """Transforms used by the native-resolution CIFAR experiment."""
    train_transform = transforms.Compose([
        transforms.Pad(4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_transform, test_transform


def create_transforms_224():
    """Transforms used by the ImageNet-style 224px experiment."""
    norm = transforms.Normalize(MEAN, STD)
    bicubic = transforms.InterpolationMode.BICUBIC
    train_transform = transforms.Compose([
        transforms.Resize(224, interpolation=bicubic),
        transforms.RandomCrop(224, padding=28),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        norm,
    ])
    test_transform = transforms.Compose([
        transforms.Resize(224, interpolation=bicubic),
        transforms.ToTensor(),
        norm,
    ])
    return train_transform, test_transform

train_transform_32, test_transform_32 = create_transforms_32()
train_transform_224, test_transform_224 = create_transforms_224()

In [ ]:
raw_dataset = datasets.CIFAR100(
    root=DATASET_PATH,
    train=True,
    download=False,
    transform=None,
)

random_index = random.randrange(len(raw_dataset))
image, label = raw_dataset[random_index]
class_name = raw_dataset.classes[label]
print(f"Random index: {random_index}")
print(f"Label: {label} ({class_name})")
print(f"Original image size: {image.size}")

In [ ]:
def to_display_image(value):
    """Convert a PIL image or normalized tensor to an RGB numpy image."""
    if not isinstance(value, torch.Tensor):
        return np.asarray(value)
    value = value.detach().cpu().float()
    value = value * torch.tensor(STD).view(3, 1, 1)
    value = value + torch.tensor(MEAN).view(3, 1, 1)
    value = value.clamp(0, 1)
    return value.permute(1, 2, 0).numpy()

def show_comparison(original, title, train_32, test_32, train_224, test_224):
    figure, axes = plt.subplots(1, 5, figsize=(20, 4))
    images = [original, train_32, test_32, train_224, test_224]
    titles = [
        "Original 32x32",
        "Pipeline 1 - train 32x32",
        "Pipeline 1 - test 32x32",
        "Pipeline 2 - train 224x224",
        "Pipeline 2 - test 224x224",
    ]
    for axis, current, current_title in zip(axes, images, titles):
        axis.imshow(to_display_image(current))
        axis.set_title(current_title)
        axis.axis("off")
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

show_comparison(
    image,
    f"Class {label}: {class_name}",
    train_transform_32(image),
    test_transform_32(image),
    train_transform_224(image),
    test_transform_224(image),
)

The training transforms are stochastic. The next cell applies each training pipeline several times to the same image, making the random crop and horizontal flip visible.

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(18, 8))
for column in range(5):
    axes[0, column].imshow(to_display_image(train_transform_32(image)))
    axes[0, column].set_title(f"Pipeline 1 / sample {column + 1}")
    axes[0, column].axis("off")
    axes[1, column].imshow(to_display_image(train_transform_224(image)))
    axes[1, column].set_title(f"Pipeline 2 / sample {column + 1}")
    axes[1, column].axis("off")
figure.suptitle("Multiple stochastic training-transform realizations")
figure.tight_layout()
plt.show()

The first pipeline preserves the native CIFAR spatial scale (`32x32`). The second pipeline enlarges the image with bicubic interpolation and produces `224x224` inputs, which is a substantially different input distribution for the network.